In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
            # data_dft_d3bj = data_d3bj
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero

        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = (
                                data_scf[col[0]] - data_cc[col[0]]
                            )
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol")
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

# print("Summary")
# display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
# print("MAE")
# display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 12.609903921023943 kcal/mol, 116 in 1424849_W4_11
  -1 * W4_11-p4  -1 * -13.835936387768015  4 * W4_11-p  4 * -0.3065081166860182
Top 2 AI: 5.341987040214008 kcal/mol, 136 in 1424849_W4_11
  -1 * W4_11-foof  -1 * -4.328057378443191  2 * W4_11-f  2 * 0.6256503599724965  2 * W4_11-o  2 * -0.11868552908708807
Top 3 AI: 5.219463497924153 kcal/mol, 46 in 1424849_W4_11
  -1 * W4_11-alf3  -1 * -3.2680325650726445  1 * W4_11-al  1 * 0.07447985294857062  3 * W4_11-f  3 * 0.6256503599724965
Top 4 AI: 4.821493316645501 kcal/mol, 135 in 1424849_W4_11
  -1 * W4_11-cloo  -1 * -4.751233361486811  1 * W4_11-cl  1 * 0.30763101333286613  2 * W4_11-o  2 * -0.11868552908708807
Top 5 AI: 4.2865408250218024 kcal/mol, 63 in 1424849_W4_11
  -1 * W4_11-sif  -1 * -3.7919159706216305  1 * W4_11-si  1 * -0.13102550557232462  1 * W4_11-f  1 * 0.6256503599724965
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.61089226667

/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.

data_path   1424849                                                      \
Disp type        AI        DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       2.002477  13.710857  1.937471  13.77654  1.842607  13.694016   
sub2       5.510109   6.612164  5.571708  6.298111  5.679644    6.47837   
sub3       2.617318   6.261376  2.896715  6.594448   3.05189   6.758425   
sub4        1.93524   3.272197  1.534615  3.409243  2.011931   4.046436   
sub5       1.626727   1.321288  1.363798  0.843797  1.385881   0.752668   

data_path            
Disp type Processed  
sub1        18 / 18  
sub2          7 / 9  
sub3          7 / 7  
sub4        11 / 12  
sub5          8 / 9

wtmad_1


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        3.239113   7.420922   2.890775     7.2373   2.735643   7.146911   
sub2       13.142366   12.53052  12.278113  11.460648  11.833095  10.570461   
sub3        4.832263   6.984991   5.141811   7.310673   5.329177   7.384597   
sub4       11.556831  11.029773   7.213895   7.031697   8.566174   9.035101   
sub5       15.335367  11.571826  12.618257   7.156504  12.856074    6.23008   

data_path            
Disp type Processed  
sub1        18 / 18  
sub2          7 / 9  
sub3          7 / 7  
sub4        11 / 12  
sub5          8 / 9

wtmad_2


data_path   1424849                                                     \
Disp type        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       1.438234  4.040696  1.327334  4.010375  1.278351   3.992288   
sub2       3.339073  3.635331  2.947971  3.094352  2.802053   2.808342   
sub3       1.303233  2.611064  1.410302   2.74261  1.477991   2.806357   
sub4       4.683102  4.159493  3.487294  3.039102  4.435006    4.35445   
sub5       6.331811   4.66236  5.508358  3.059454  5.665142   2.701503   

data_path            
Disp type Processed  
sub1        18 / 18  
sub2          7 / 9  
sub3          7 / 7  
sub4        11 / 12  
sub5          8 / 9

Summary of Subset
